# Bronze Layer
###Satellite Intelligence Assignment

This notebook loads the raw source files into the Bronze layer.

Bronze layer keeps the data in its original form so we can always trace back to the source if needed..

In [0]:
#Load Import Libraries

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, TimestampType, FloatType
import pyspark.sql.functions as F

In [0]:
#Define Catalog
catalog_name = 'carnot-catalog'

In [0]:
# define Brand Raw table path

metadata_raw_data_path =  "/Volumes/carnot-catalog/source-data/rawdata/parcel_metadata.csv"
metadata_raw_data_path

'/Volumes/carnot-catalog/source-data/rawdata/parcel_metadata.csv'

In [0]:
# define Brand Raw table path

parcel_readings_raw_data_path =  "/Volumes/carnot-catalog/source-data/rawdata/parcel_readings.csv"
parcel_readings_raw_data_path

'/Volumes/carnot-catalog/source-data/rawdata/parcel_readings.csv'

In [0]:
#define schema

#Using a fixed schema helps avoid wrong datatype detection while reading files



parcel_metadata_schema = StructType([
    StructField("parcel_id", StringType(), True),
    StructField("mill_id", StringType(), True),
    StructField("crop_type", StringType(), True),
    StructField("sowing_date", StringType(), True),
    StructField("area_hectares", FloatType(), True)
])
#define schema
parcel_readings_schema = StructType([
    StructField("parcel_id", StringType(), True),
    StructField("date", StringType(), True),
    StructField("ndvi_value", FloatType(), True),
    StructField("temperature_c", FloatType(), True),
    StructField("rainfall_mm", FloatType(), True),
    StructField("sensor_status", StringType(), True)
])


In [0]:
#Read Raw Data and write to Bronze Table

parcel_metadata_df = spark.read.option('header',"true").option("delimiter",",").schema(parcel_metadata_schema).csv(metadata_raw_data_path)


# add MetaData Columns

parcel_metadata_df = parcel_metadata_df.withColumn("_source_file", F.col("_metadata.file_path")) \
       .withColumn("ingested_at", F.current_timestamp())

display(parcel_metadata_df)  


parcel_id,mill_id,crop_type,sowing_date,area_hectares,_source_file,ingested_at
PARCEL_001,MILL_NORTH,sugarcane,2026-02-10,9.03,dbfs:/Volumes/carnot-catalog/source-data/rawdata/parcel_metadata.csv,2026-05-18T18:45:01.488Z
PARCEL_002,MILL_SOUTH,sugarcane,2026-01-16,8.97,dbfs:/Volumes/carnot-catalog/source-data/rawdata/parcel_metadata.csv,2026-05-18T18:45:01.488Z
PARCEL_003,MILL_NORTH,soybean,2026-02-13,5.35,dbfs:/Volumes/carnot-catalog/source-data/rawdata/parcel_metadata.csv,2026-05-18T18:45:01.488Z
PARCEL_004,MILL_NORTH,sugarcane,2026-01-02,3.18,dbfs:/Volumes/carnot-catalog/source-data/rawdata/parcel_metadata.csv,2026-05-18T18:45:01.488Z
PARCEL_005,MILL_NORTH,soybean,2026-02-08,2.79,dbfs:/Volumes/carnot-catalog/source-data/rawdata/parcel_metadata.csv,2026-05-18T18:45:01.488Z
PARCEL_006,MILL_WEST,sugarcane,2026-02-11,5.67,dbfs:/Volumes/carnot-catalog/source-data/rawdata/parcel_metadata.csv,2026-05-18T18:45:01.488Z
PARCEL_007,MILL_NORTH,sugarcane,2026-01-18,8.53,dbfs:/Volumes/carnot-catalog/source-data/rawdata/parcel_metadata.csv,2026-05-18T18:45:01.488Z
PARCEL_008,MILL_EAST,sugarcane,2026-01-22,2.98,dbfs:/Volumes/carnot-catalog/source-data/rawdata/parcel_metadata.csv,2026-05-18T18:45:01.488Z
PARCEL_009,MILL_EAST,sugarcane,2026-02-18,1.57,dbfs:/Volumes/carnot-catalog/source-data/rawdata/parcel_metadata.csv,2026-05-18T18:45:01.488Z
PARCEL_010,MILL_EAST,sugarcane,2026-01-07,7.44,dbfs:/Volumes/carnot-catalog/source-data/rawdata/parcel_metadata.csv,2026-05-18T18:45:01.488Z


In [0]:
#Read Raw Data and write to Bronze Table

parcel_readings_df = spark.read.option('header',"true").option("delimiter",",").schema(parcel_readings_schema).csv(parcel_readings_raw_data_path)


# add MetaData Columns

parcel_readings_df = parcel_readings_df.withColumn("_source_file", F.col("_metadata.file_path")) \
       .withColumn("ingested_at", F.current_timestamp())

display(parcel_readings_df.limit(5))  


parcel_id,date,ndvi_value,temperature_c,rainfall_mm,sensor_status,_source_file,ingested_at
PARCEL_018,16/05/2026,0.595,15.4,0.0,error,dbfs:/Volumes/carnot-catalog/source-data/rawdata/parcel_readings.csv,2026-05-18T18:45:10.244Z
PARCEL_014,2026-01-27,0.457,27.6,0.0,OK,dbfs:/Volumes/carnot-catalog/source-data/rawdata/parcel_readings.csv,2026-05-18T18:45:10.244Z
PARCEL_006,2026-05-17,0.497,25.8,0.0,OK,dbfs:/Volumes/carnot-catalog/source-data/rawdata/parcel_readings.csv,2026-05-18T18:45:10.244Z
PARCEL_004,10/02/2026,0.81,25.0,0.0,OK,dbfs:/Volumes/carnot-catalog/source-data/rawdata/parcel_readings.csv,2026-05-18T18:45:10.244Z
PARCEL_002,2026-01-17,0.269,33.2,0.0,OK,dbfs:/Volumes/carnot-catalog/source-data/rawdata/parcel_readings.csv,2026-05-18T18:45:10.244Z


In [0]:
#Save output File for parcel_metadata

parcel_metadata_df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"`{catalog_name}`.bronze.Bronze_Parcel_Metadata")

In [0]:
#Save output File for parcel_readings

parcel_readings_df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"`{catalog_name}`.bronze.Bronze_Parcel_Readings")